# Week 1.4: Prompt Engineering — now with a real AI behind it
### Plain-English edition — no coding required

Until now your twin has been maths on your own machine. From this notebook on, it is
**powered by a real language model** running on Groq's servers.

That changes what this lesson can be. You will not read about prompting techniques —
you will **send the same question two different ways and watch the answers differ**.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## Before you start: your free key
This notebook talks to a real model, so it needs a **free Groq API key**.

1. Go to **console.groq.com** and sign up (free, no card needed)
2. Create an API key and copy it
3. Back in Colab, click the **🔑 key icon** in the left sidebar
4. Add a secret named exactly **`GROQ_API_KEY`** and paste your key in
5. Switch **Notebook access** on

> **Never paste your key into a cell or a box.** Anyone who sees your notebook —
> or a screenshot, or a video of your screen — would have your key and your bill.
> The Secrets panel keeps it out of sight.

## What you will learn
- **System instructions** — the standing rules that decide who your twin is
- **Zero-shot vs few-shot** — telling versus showing, side by side, for real
- **Chain of thought** — making it slow down, and watching the answer improve
- **Temperature** — the dial between predictable and creative
- **The prompt stack** — what is really sent every single turn

In [ ]:
#@title ▶ Step 1 — Wake up my twin (connect it to the AI) { display-mode: "form" }
#@markdown Nothing to fill in. Press play and I will check your key and say hello to the model.

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

c, k = _client()
if c is not None:
    success("Connected. Your key: " + mask(k))
    note("Only the first and last few characters are ever shown - never the whole key.")
    print("")
    print("Saying hello to the model to be sure it really works...")
    print("")
    reply = llm([{"role": "user", "content": "Say hello in one short sentence."}], temperature=0.0, max_tokens=40)
    if reply:
        show("THE MODEL REPLIED", reply)
        print("")
        print("Your twin now has a brain. Everything below sends real messages to it.")
        print("")
        print("So what? Three things made that work: a key that proves who you are,")
        print("a model name, and your message. That is all any AI service needs.")

---
## Part 1: The standing rules

Before anyone types a word, you hand the model a brief: who it is, what it knows,
what it must never do, and how it should sound. That brief is the **system instruction**,
and it is the single biggest lever you have.

In [ ]:
#@title ▶ Step 2 — Write my twin's rules and test them live { display-mode: "form" }
#@markdown **ROLE** — who is it?
role = "a digital twin of Ama, a 17-year-old student from Accra" #@param {type:"string"}
#@markdown **KNOWLEDGE** — what may it talk about?
knowledge = "Ama's school subjects, her football training, and her daily routine" #@param {type:"string"}
#@markdown **CONSTRAINTS** — what must it never do?
constraints = "never invent facts about Ama, and never claim to be a real human" #@param {type:"string"}
#@markdown **TONE** — how should it sound?
tone = "friendly and direct, like a classmate" #@param {type:"string"}
#@markdown Now ask your twin something:
my_question = "Who are you, and what did you do on Saturday?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

parts = {"ROLE": role, "KNOWLEDGE": knowledge, "CONSTRAINTS": constraints, "TONE": tone}
empty = [k for k, v in parts.items() if not v.strip()]
if empty:
    problem("These parts are still empty: " + ", ".join(empty))
    note("Each line does a different job. Fill them all in and press play again.")
else:
    brief = chr(10).join(k + ": " + v.strip() for k, v in parts.items())
    globals()["_SYSTEM"] = brief
    show("YOUR SYSTEM INSTRUCTION (sent before every single message)", brief)
    print("")
    answer = llm([{"role": "system", "content": brief},
                  {"role": "user", "content": my_question.strip()}], temperature=0.4)
    if answer:
        show("YOUR TWIN ANSWERS", answer)
        print("")
        print("Now compare it with the SAME question and no rules at all:")
        print("")
        plain = llm([{"role": "user", "content": my_question.strip()}], temperature=0.4)
        if plain:
            show("THE SAME MODEL, WITH NO RULES", plain)
            print("")
            print("So what? Same model, same question, completely different answer.")
            print("You did not retrain anything. You just told it who to be.")
            note("Try changing ROLE to a farmer, a nurse, a teacher - then run this again.")

---
## Part 2: Telling vs showing

**Zero-shot** describes the task in words. **Few-shot** shows two worked examples first.
The step below sends both to the real model so you can judge the difference yourself.

In [ ]:
#@title ▶ Step 3 — Ask the same thing two ways and compare { display-mode: "form" }
#@markdown The job you want done:
my_task = "Pull out the likes and dislikes from the sentence." #@param {type:"string"}
#@markdown The sentence to work on:
my_input = "I love football and jollof rice but I hate waking up early" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

zero = my_task.strip() + chr(10) + chr(10) + "Input: " + my_input.strip() + chr(10) + "Output:"
examples = ("Example 1" + chr(10) +
            "Input: I enjoy swimming but I cannot stand loud music" + chr(10) +
            'Output: {"likes": ["swimming"], "dislikes": ["loud music"]}' + chr(10) + chr(10) +
            "Example 2" + chr(10) +
            "Input: Reading is lovely, crowds are not" + chr(10) +
            'Output: {"likes": ["reading"], "dislikes": ["crowds"]}' + chr(10) + chr(10))
few = my_task.strip() + chr(10) + chr(10) + examples + "Now you:" + chr(10) + "Input: " + my_input.strip() + chr(10) + "Output:"
print("Sending the SAME request twice - once plain, once with examples.")
print("")
a = llm([{"role": "user", "content": zero}], temperature=0.0)
b = llm([{"role": "user", "content": few}], temperature=0.0)
if a and b:
    show("1. ZERO-SHOT - described the job in words", a)
    print("")
    show("2. FEW-SHOT - showed two worked examples first", b)
    print("")
    tidy = lambda t: t.strip().startswith("{") and t.strip().endswith("}")
    print("Does the answer come back as clean data your code could use?")
    (success if tidy(a) else problem)("   zero-shot: " + ("yes" if tidy(a) else "no - there is chatter around it"))
    (success if tidy(b) else problem)("   few-shot:  " + ("yes" if tidy(b) else "no - there is chatter around it"))
    print("")
    print("So what? Both answers may be CORRECT, but only one is usable without")
    print("extra cleaning. If the shape of the answer matters, show - do not tell.")
    note("Those two examples cost a handful of extra tokens. Almost always worth it.")

---
## Part 3: Making it slow down

Ask something that needs several steps and a model will often blurt out the first
plausible answer. Four words — **"think step by step"** — change that.

In [ ]:
#@title ▶ Step 4 — Watch "think step by step" change the answer { display-mode: "form" }
#@markdown A question that takes more than one step to work out:
question = "Ama studies 2 hours each weekday and 4 hours on Saturday. She rests on Sunday. How many hours does she study in a week?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

print("Sending the SAME question twice - once rushed, once asked to think.")
print("")
fast = llm([{"role": "user", "content": question.strip() + chr(10) +
             "Reply with the final answer only. No working."}], temperature=0.0, max_tokens=60)
slow = llm([{"role": "user", "content": question.strip() + chr(10) +
             "Think step by step, then give the final answer."}], temperature=0.0, max_tokens=400)
if fast and slow:
    show("1. RUSHED - answer only", fast)
    print("")
    show("2. ASKED TO THINK - working shown", slow)
    print("")
    print("Read both. Do they agree?")
    print("")
    print("So what? Those middle sentences are not decoration. The model reads")
    print("its own working as it writes, so each step steadies the next.")
    print("And when it shows the working, YOU can spot where it went wrong -")
    print("which you can never do with a bare answer.")
    note("Try a harder question - more steps, bigger numbers - and run it again.")

---
## Part 4: The creativity dial

**Temperature** decides how adventurous the model is. Near 0 it picks the most likely
word every time. Near 1 it takes risks. Same prompt, very different results.

In [ ]:
#@title ▶ Step 5 — Turn the creativity dial up and down { display-mode: "form" }
#@markdown Something with many possible good answers:
prompt = "Suggest a name for Ama's football team in three words or fewer." #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

print("Asking the same thing twice at each setting.")
print("")
for temp, label in [(0.0, "TEMPERATURE 0.0 - predictable"), (1.0, "TEMPERATURE 1.0 - adventurous")]:
    outs = []
    for _ in range(2):
        r = llm([{"role": "user", "content": prompt.strip()}], temperature=temp, max_tokens=40)
        if r: outs.append(r.replace(chr(10), " ").strip())
    if len(outs) == 2:
        print(label)
        print("   try 1: " + outs[0][:70])
        print("   try 2: " + outs[1][:70])
        if temp == 0.0:
            print("   " + ("-> identical, as expected" if outs[0] == outs[1] else "-> nearly identical"))
        else:
            print("   " + ("-> different, as expected" if outs[0] != outs[1] else "-> same this time, but it varies"))
        print("")
print("So what? Low temperature for anything that must be consistent:")
print("extracting data, following a format, answering factual questions.")
print("High temperature for brainstorming, names, stories.")
note("A twin answering questions about your life wants LOW temperature. Inventing is not a feature there.")

---
## Part 5: The whole parcel, and memory

The model forgets you the moment it replies. A conversation only exists because
**you** send the entire history back every single time.

In [ ]:
#@title ▶ Step 6 — Have a real conversation (run me again and again) { display-mode: "form" }
#@markdown Say something to your twin. Press play, change this box, press play again.
what_i_say = "My name is Ama and I play football on Saturdays." #@param {type:"string"}
#@markdown Tick to wipe the conversation and start fresh:
start_over = False #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

if start_over or "_CHAT" not in globals():
    globals()["_CHAT"] = [{"role": "system", "content": twin_rules()}]
    note("Starting a fresh conversation.")
    print("")
chat = globals()["_CHAT"]
if not what_i_say.strip():
    problem("Type something to say first.")
else:
    chat.append({"role": "user", "content": what_i_say.strip()})
    reply = llm(chat, temperature=0.4)
    if reply is None:
        chat.pop()
    else:
        chat.append({"role": "assistant", "content": reply})
        turns = sum(1 for m in chat if m["role"] == "user")
        print("THE CONVERSATION SO FAR (" + str(turns) + " turn" + ("s" if turns != 1 else "") + "):")
        print("")
        for m in chat:
            if m["role"] == "system": continue
            who = "You " if m["role"] == "user" else "Twin"
            for i, line in enumerate(m["content"].split(chr(10))):
                print(("   " + who + ": " if i == 0 else "         ") + line)
        u = globals().get("_LAST_USAGE")
        if u:
            print("")
            note("This turn sent " + str(u.prompt_tokens) + " tokens - the WHOLE conversation plus the rules, " +
                 "not just your last line. It replied with " + str(u.completion_tokens) + ".")
        print("")
        if turns == 1:
            print("Now change the box above to 'What is my name?' and press play again.")
            print("It will remember - because the history travels with every request.")
        else:
            print("So what? The model has no memory of its own. You are re-sending the")
            print("entire chat every time, and paying for all of it every time.")
            note("Watch the token count climb. That is exactly the problem Week 2 solves.")

### 🎯 Exercise: grade your own prompt, then run it

Write a prompt, have it marked against the five things that matter, then send it
to your twin for real and see whether the score was fair.

In [ ]:
#@title ▶ Step 7 — Mark my prompt, then actually send it { display-mode: "form" }
#@markdown Write a prompt you would really use. Try a weak one first, then improve it.
my_prompt = "Tell me about my week" #@param {type:"string"}
#@markdown Send it to the twin as well as marking it?
also_send_it = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

MODEL = "llama-3.3-70b-versatile"
def _client():
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    if not k or not k.strip():
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    return Groq(api_key=k.strip()), k.strip()
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=400, quiet=False):
    c, k = _client()
    if c is None: return None
    try:
        r = c.chat.completions.create(model=MODEL, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            problem("The server rejected your key."); note("Check the key saved in Colab Secrets is still valid.")
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    out = r.choices[0].message.content.strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

p = my_prompt.strip(); low = p.lower()
if len(p) < 5:
    problem("Please write a prompt first.")
else:
    checks = [
      ("Says who the model should be",
       any(k in low for k in ["you are", "act as", "role", "pretend", "as a", "twin"]),
       "Start with who it is: 'You are a digital twin of...'"),
      ("Says what to produce",
       any(k in low for k in ["list", "summar", "explain", "write", "predict", "extract", "compare", "suggest", "plan"]),
       "Name the output: a list, a summary, a plan, an explanation"),
      ("Gives something concrete to work with",
       any(ch.isdigit() for ch in p) or len(p.split()) > 18,
       "Add the real details - dates, numbers, the text to work on"),
      ("Sets a limit or a format",
       any(k in low for k in ["json", "bullet", "sentence", "word", "max", "under", "three", "two", "five", "short", "format", "table"]),
       "Say how long or what shape: 'three bullets', 'one sentence', 'as JSON'"),
      ("Says what NOT to do",
       any(k in low for k in ["do not", "don't", "never", "avoid", "without", "only"]),
       "Add a boundary: 'do not invent anything you were not told'"),
    ]
    passed = sum(1 for _, ok, _ in checks if ok)
    show("YOUR PROMPT", p)
    print("")
    print("Marked against five things that matter:")
    print("")
    for name, ok, _ in checks:
        (success if ok else problem)(("PASS  " if ok else "MISS  ") + name)
    print("")
    print("Score: " + str(passed) + " out of 5")
    misses = [fix for name, ok, fix in checks if not ok]
    if misses:
        print("")
        print("To improve it:")
        for m in misses: print("   - " + m)
    else:
        print("")
        success("Nothing missing. That is a prompt a model can actually follow.")
    if also_send_it:
        print("")
        print("Sending it to your twin now...")
        print("")
        out = llm([{"role": "system", "content": twin_rules()}, {"role": "user", "content": p}], temperature=0.3)
        if out:
            show("WHAT YOUR TWIN MADE OF IT", out)
            print("")
            if passed <= 2:
                print("A low score usually shows up as a vague or invented answer.")
                print("Look closely: did it make anything up to fill the gaps you left?")
            else:
                print("A specific prompt gets a specific answer. Compare this with what")
                print("a one-line version of the same request would have produced.")
    print("")
    print("So what? A vague prompt does not get a vague answer.")
    print("It gets a confident, specific answer to a question you never asked.")

---
## Summary & Key Takeaways

1. **Your twin is live.** A key, a model name and a message — that is all any AI service needs.
2. **System instructions** decide who it is. One sentence changes everything; the constraints line is the one people skip.
3. **Few-shot beats zero-shot** whenever the shape of the answer matters. Show, don't tell.
4. **"Think step by step"** turns a guess into working you can check.
5. **Temperature** — low for facts and formats, high for brainstorming.
6. **No memory.** The whole conversation is re-sent, and re-paid for, every turn.
7. **Grade your prompts** — who, what, which details, what shape, what to avoid.

> **Keep your key in Secrets, never in a cell.** A key in a screenshot, a shared
> notebook, or a screen recording is a key someone else can spend.

### Next up
Week 2: cleaning messy text, staying inside the token limit, and giving your twin
real knowledge to draw on.